In [1]:
import sys
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))


In [2]:
from src.process_weather import weather_processing

file_path = Path("../data/processed/era5_merged.nc")

weather = weather_processing(file_path)

print(weather)

Years:
[2021 2022 2023 2024 2025 2026]

Raw radiation ranges:
ssrd: min=-1.60, max=3737216.00
ssr: min=0.00, max=3010624.00
strd: min=1087042.00, max=1773911.00
str: min=-818797.00, max=-3173.00
fdir: min=0.00, max=3168256.00

Flux ranges:
ssrd_flux: min=-0.00, max=1038.12
ssr_flux: min=0.00, max=836.28
strd_flux: min=301.96, max=492.75
str_flux: min=-227.44, max=-0.88
fdir_flux: min=0.00, max=880.07
<xarray.Dataset> Size: 904B
Dimensions:     (time: 24)
Coordinates:
  * time        (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    valid_time  (time) datetime64[ns] 192B ...
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
    latitude    float64 8B 23.0
    longitude   float64 8B 72.5
Data variables:
    ssrd_flux   (time) float32 96B 0.0 0.0 60.48 281.0 529.9 ... 0.0 0.0 0.0 0.0
    ssr_flux    (time) float32 96B 1e-15 1e-15 47.04 221.2 ... 1e-15 1e-15 1e-15
    strd_flux   (time) float32 96B 315.6 310.4 307.0 309.7 ...

In [3]:
sample = weather.isel(
    latitude=1,
    longitude=1,
    time=slice(0, 24)
)

print(
    sample[
        [
            "ssrd",
            "ssr",
            "strd",
            "str",
            "fdir"
        ]
    ]
)

<xarray.Dataset> Size: 904B
Dimensions:     (time: 24)
Coordinates:
  * time        (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    valid_time  (time) datetime64[ns] 192B ...
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
    latitude    float64 8B 23.0
    longitude   float64 8B 72.5
Data variables:
    ssrd        (time) float32 96B 0.0 0.0 2.177e+05 1.012e+06 ... 0.0 0.0 0.0
    ssr         (time) float32 96B 3.6e-12 3.6e-12 1.693e+05 ... 3.6e-12 3.6e-12
    strd        (time) float32 96B 1.136e+06 1.117e+06 ... 1.17e+06 1.164e+06
    str         (time) float32 96B -3.468e+05 -3.429e+05 ... -3.075e+05
    fdir        (time) float32 96B 0.0 0.0 1.123e+05 7.11e+05 ... 0.0 0.0 0.0
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:

In [4]:
import pandas as pd
import numpy as np
from pvlib.solarposition import get_solarposition

lat = 23.0
lon = 72.5

times = pd.DatetimeIndex(weather.time.values).tz_localize("UTC")

solar_position = get_solarposition(
    times,
    latitude=lat,
    longitude=lon,
    method="nrel_numpy"
)

solar_position.head()

,apparent_zenith,zenith,apparent_elevation,elevation,azimuth,equation_of_time
2021-04-01 00:00:00+00:00,105.072986,105.072986,-15.072986,-15.072986,78.264494,-3.941129
2021-04-01 01:00:00+00:00,91.417455,91.417455,-1.417455,-1.417455,84.442397,-3.928705
2021-04-01 02:00:00+00:00,77.547854,77.621398,12.452146,12.378602,90.256552,-3.916285
2021-04-01 03:00:00+00:00,63.799701,63.833674,26.200299,26.166326,96.413683,-3.903869
2021-04-01 04:00:00+00:00,50.214479,50.234665,39.785521,39.765335,103.871324,-3.891456


In [5]:
from src.cos_solar import calculate_cos_solar_zenith

weather["cos_theta"] = calculate_cos_solar_zenith(weather)

print(weather["cos_theta"])

<xarray.DataArray 'cos_theta' (time: 13104, latitude: 3, longitude: 3)> Size: 472kB
array([[[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ]],

       [[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ]],

       [[0.21043193, 0.21434943, 0.21826346],
        [0.21044575, 0.21437056, 0.2182919 ],
        [0.21045555, 0.2143876 , 0.21831615]],

       ...,

       [[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ]],

       [[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ]],

       [[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ]]],
      shape=(13104, 3, 3), dtype=float32)
Coordinates:
  * time

In [6]:
from src.process_weather import radiation_processing

weather = radiation_processing(weather)

print(
    weather[
        [
            "I_sw",
            "D_sw",
            "R_sw",
            "U_lw",
            "D_lw"
        ]
    ]
)

<xarray.Dataset> Size: 3MB
Dimensions:     (latitude: 3, longitude: 3, time: 13104)
Coordinates:
  * latitude    (latitude) float64 24B 23.25 23.0 22.75
  * longitude   (longitude) float64 24B 72.25 72.5 72.75
  * time        (time) datetime64[ns] 105kB 2021-04-01 ... 2026-06-30T23:00:00
    valid_time  (time) datetime64[ns] 105kB ...
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Data variables:
    I_sw        (latitude, longitude, time) float32 472kB 0.0 0.0 ... 0.0 0.0
    D_sw        (latitude, longitude, time) float32 472kB 0.0 0.0 ... 0.0 0.0
    R_sw        (latitude, longitude, time) float32 472kB -1e-15 ... -1e-15
    U_lw        (latitude, longitude, time) float32 472kB 406.0 400.3 ... 474.8
    D_lw        (latitude, longitude, time) float32 472kB 310.5 306.1 ... 443.3
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts

In [7]:
from src.mrt import calculate_mrt

weather = calculate_mrt(weather)

print(weather["mrt"])

<xarray.DataArray 'mrt' (latitude: 3, longitude: 3, time: 13104)> Size: 472kB
array([[[ 8.785309 ,  7.7893066, 16.374084 , ..., 27.387024 ,
         28.02893  , 27.212677 ],
        [ 8.334778 ,  7.3547974, 16.40506  , ..., 27.01236  ,
         27.451996 , 26.764038 ],
        [ 7.868805 ,  6.934326 , 16.415009 , ..., 26.655945 ,
         26.766449 , 26.29367  ]],

       [[10.269684 ,  9.141846 , 17.248383 , ..., 27.69983  ,
         28.37265  , 27.699615 ],
        [ 9.863403 ,  8.741577 , 17.263947 , ..., 27.296722 ,
         27.777557 , 27.18808  ],
        [ 9.719788 ,  8.57782  , 17.481567 , ..., 26.84845  ,
         27.067474 , 26.526215 ]],

       [[11.725189 , 10.6223755, 18.381897 , ..., 28.090546 ,
         28.58258  , 28.007172 ],
        [10.922485 ,  9.757446 , 17.92096  , ..., 27.470001 ,
         27.967224 , 27.46112  ],
        [10.826691 ,  9.636627 , 18.10196  , ..., 26.97757  ,
         27.311218 , 26.819214 ]]], shape=(3, 3, 13104), dtype=float32)
Coordinates:
  *

In [8]:
mrt_sample = weather.isel(
    latitude=1,
    longitude=1,
    time=slice(0, 24)
)

print(
    mrt_sample[
        [
            "tdb",
            "mrt"
        ]
    ]
)

<xarray.Dataset> Size: 616B
Dimensions:     (time: 24)
Coordinates:
  * time        (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    valid_time  (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    number      int64 8B 0
    step        timedelta64[ns] 8B 00:00:00
    surface     float64 8B 0.0
    latitude    float64 8B 23.0
    longitude   float64 8B 72.5
Data variables:
    tdb         (time) float32 96B 23.87 23.47 24.34 24.4 ... 22.86 22.37 22.12
    mrt         (time) float32 96B 9.863 8.742 17.26 35.29 ... 11.06 10.58 10.29
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-30T08:01 GRIB to CDM+CF via cfgrib-0.9.1...


In [9]:
print("MRT range:")
print(
    "min =", float(weather["mrt"].min(skipna=True)),
    "max =", float(weather["mrt"].max(skipna=True))
)

print("\nMissing MRT:")
print(int(weather["mrt"].isnull().sum()))

MRT range:
min = 6.934326171875 max = 68.97476196289062

Missing MRT:
0


In [10]:
mrt_diff = weather["mrt"] - weather["tdb"]

print("MRT - TDB:")
print(
    "min =", float(mrt_diff.min(skipna=True)),
    "max =", float(mrt_diff.max(skipna=True)),
    "mean =", float(mrt_diff.mean(skipna=True))
)

MRT - TDB:
min = -20.174468994140625 max = 34.333740234375 mean = 5.748848915100098


In [11]:
sample = weather.isel(
    latitude=1,
    longitude=1,
    time=slice(0, 24)
)

print(
    sample[
        [
            "tdb",
            "mrt",
            "I_sw",
            "D_sw",
            "R_sw",
            "U_lw",
            "D_lw"
        ]
    ]
)

<xarray.Dataset> Size: 1kB
Dimensions:     (time: 24)
Coordinates:
  * time        (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    valid_time  (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    number      int64 8B 0
    step        timedelta64[ns] 8B 00:00:00
    surface     float64 8B 0.0
    latitude    float64 8B 23.0
    longitude   float64 8B 72.5
Data variables:
    tdb         (time) float32 96B 23.87 23.47 24.34 24.4 ... 22.86 22.37 22.12
    mrt         (time) float32 96B 9.863 8.742 17.26 35.29 ... 11.06 10.58 10.29
    I_sw        (time) float32 96B 0.0 0.0 6.685 87.09 266.9 ... 0.0 0.0 0.0 0.0
    D_sw        (time) float32 96B 0.0 0.0 29.3 83.56 112.6 ... 0.0 0.0 0.0 0.0
    R_sw        (time) float32 96B -1e-15 -1e-15 13.44 ... -1e-15 -1e-15 -1e-15
    U_lw        (time) float32 96B 411.9 405.7 405.6 433.8 ... 412.8 409.9 408.6
    D_lw        (time) float32 96B 315.6 310.4 307.0 309.7 ... 327.1 324.9 323.2
Attributes:
    GRIB_edition:  

In [12]:
# UTCI calculation
# limit_inputs=False is used because 777 valid ERA5 observations
# have 10-m wind speeds below the pythermalcomfort applicability
# threshold of 0.5 m/s. Wind values are not artificially clipped.
from src.utci import calculate_utci

weather = calculate_utci(weather)

print(weather["utci"])

<xarray.DataArray 'utci' (latitude: 3, longitude: 3, time: 13104)> Size: 943kB
array([[[16.28958017, 15.89641737, 19.24442176, ..., 30.45069796,
         30.30762469, 30.18680306],
        [15.60829835, 14.96960214, 18.763403  , ..., 30.94109443,
         31.05287091, 30.69928006],
        [15.08175576, 14.66462097, 19.13166212, ..., 30.76175095,
         31.27084678, 31.27216557]],

       [[18.94719514, 18.40508205, 20.58163797, ..., 31.33087889,
         31.26433904, 31.04447671],
        [18.08911456, 17.36891293, 19.88886601, ..., 31.56559014,
         31.93611132, 31.57784229],
        [17.68426636, 16.56095656, 19.83239342, ..., 30.87420673,
         31.41587366, 31.68084283]],

       [[20.41129436, 20.10086047, 21.69814458, ..., 31.77823988,
         31.77358773, 31.67068334],
        [19.5994434 , 19.00520482, 21.38247999, ..., 31.55528622,
         32.19731328, 32.2601461 ],
        [19.28126566, 18.17812792, 21.61236335, ..., 30.8503725 ,
         31.47058289, 31.83255233]]

In [13]:
print(
    "UTCI min:",
    float(weather["utci"].min(skipna=True))
)

print(
    "UTCI max:",
    float(weather["utci"].max(skipna=True))
)

print(
    "UTCI NaN:",
    int(weather["utci"].isnull().sum())
)

UTCI min: 13.948166759916814
UTCI max: 51.05730916704694
UTCI NaN: 0


In [14]:
print("tdb :", weather["tdb"].dims)
print("mrt :", weather["mrt"].dims)
print("v   :", weather["v"].dims)
print("rh  :", weather["rh"].dims)
print("utci:", weather["utci"].dims)

tdb : ('time', 'latitude', 'longitude')
mrt : ('latitude', 'longitude', 'time')
v   : ('time', 'latitude', 'longitude')
rh  : ('time', 'latitude', 'longitude')
utci: ('latitude', 'longitude', 'time')


In [15]:
# Check UTCI input validity

tdb = weather["tdb"].transpose(
    "latitude", "longitude", "time"
)

mrt = weather["mrt"]

v = weather["v"].transpose(
    "latitude", "longitude", "time"
)

rh = weather["rh"].transpose(
    "latitude", "longitude", "time"
)

print("Invalid tdb:",
      int(((tdb <= -50) | (tdb >= 50)).sum()))

print("Invalid MRT:",
      int(((mrt <= tdb - 30) | (mrt >= tdb + 70)).sum()))

print("Invalid wind:",
      int(((v <= 0.5) | (v >= 17)).sum()))

print("Invalid RH:",
      int(((rh < 0) | (rh > 100)).sum()))

Invalid tdb: 0
Invalid MRT: 0
Invalid wind: 777
Invalid RH: 0


In [16]:
invalid_utci = weather["utci"].isnull()

print(
    "Number of invalid UTCI values:",
    int(invalid_utci.sum())
)

Number of invalid UTCI values: 0


In [17]:
print(
    "Wind minimum:",
    float(weather["v"].min())
)

print(
    "Wind <= 0.5:",
    int((weather["v"] <= 0.5).sum())
)

Wind minimum: 0.012864254415035248
Wind <= 0.5: 777


In [18]:
sample = weather.isel(
    latitude=1,
    longitude=1
)

print(
    sample[
        [
            "time",
            "tdb",
            "mrt",
            "v",
            "rh",
            "utci"
        ]
    ].isel(time=slice(0, 24))
)

<xarray.Dataset> Size: 1kB
Dimensions:     (time: 24)
Coordinates:
  * time        (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    valid_time  (time) datetime64[ns] 192B 2021-04-01 ... 2021-04-01T23:00:00
    number      int64 8B 0
    step        timedelta64[ns] 8B 00:00:00
    surface     float64 8B 0.0
    latitude    float64 8B 23.0
    longitude   float64 8B 72.5
Data variables:
    tdb         (time) float32 96B 23.87 23.47 24.34 24.4 ... 22.86 22.37 22.12
    mrt         (time) float32 96B 9.863 8.742 17.26 35.29 ... 11.06 10.58 10.29
    v           (time) float32 96B 2.419 2.355 2.745 2.361 ... 2.175 2.144 2.181
    rh          (time) float32 96B 56.0 54.86 50.27 50.93 ... 76.13 88.18 92.5
    utci        (time) float64 192B 18.09 17.37 19.89 ... 19.09 19.37 19.29
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conven

In [19]:
print(
    "Central grid UTCI max:",
    float(sample["utci"].max(skipna=True))
)

print(
    "Central grid TDB max:",
    float(sample["tdb"].max(skipna=True))
)

print(
    "Central grid MRT max:",
    float(sample["mrt"].max(skipna=True))
)

Central grid UTCI max: 50.316592796088756
Central grid TDB max: 44.209442138671875
Central grid MRT max: 67.75045776367188


In [20]:
import pandas as pd

utci_df = (
    weather[
        [
            "tdb",
            "mrt",
            "v",
            "rh",
            "utci"
        ]
    ]
    .to_dataframe()
    .reset_index()
)

print(utci_df.head())
print(utci_df.shape)
print(utci_df.isna().sum())

        time  latitude  longitude        tdb        mrt         v         rh  \
0 2021-04-01     23.25      72.25  22.828949   8.785309  2.499773  50.233261   
1 2021-04-01     23.25      72.50  22.635590   8.334778  2.416389  41.109909   
2 2021-04-01     23.25      72.75  22.196136   7.868805  2.510747  44.236458   
3 2021-04-01     23.00      72.25  24.106293  10.269684  2.510639  64.983521   
4 2021-04-01     23.00      72.50  23.869965   9.863403  2.418709  55.996449   

        utci  number   step  surface valid_time  
0  16.289580       0 0 days      0.0 2021-04-01  
1  15.608298       0 0 days      0.0 2021-04-01  
2  15.081756       0 0 days      0.0 2021-04-01  
3  18.947195       0 0 days      0.0 2021-04-01  
4  18.089115       0 0 days      0.0 2021-04-01  
(117936, 12)
time          0
latitude      0
longitude     0
tdb           0
mrt           0
v             0
rh            0
utci          0
number        0
step          0
surface       0
valid_time    0
dtype: int64


In [21]:
import pandas as pd
from pathlib import Path

# Load the Day 2 UTCI output
input_path = Path("data/processed/utci_hourly.csv")

df = pd.read_csv(input_path)

# Keep only the variables needed for downstream analysis
analysis_df = df[
    [
        "time",
        "latitude",
        "longitude",
        "tdb",
        "mrt",
        "v",
        "rh",
        "utci",
    ]
].copy()

# Make sure time is actually datetime
analysis_df["time"] = pd.to_datetime(analysis_df["time"])

# Sort for reproducible downstream processing
analysis_df = analysis_df.sort_values(
    ["latitude", "longitude", "time"]
).reset_index(drop=True)

print(analysis_df.head())
print("\nShape:", analysis_df.shape)
print("\nMissing values:")
print(analysis_df.isna().sum())

                 time  latitude  longitude        tdb        mrt         v  \
0 2021-04-01 00:00:00     22.75      72.25  25.403168  11.725189  2.864268   
1 2021-04-01 01:00:00     22.75      72.25  25.126251  10.622376  2.693404   
2 2021-04-01 02:00:00     22.75      72.25  25.441132  18.381897  3.202273   
3 2021-04-01 03:00:00     22.75      72.25  25.519806  35.881530  2.855471   
4 2021-04-01 04:00:00     22.75      72.25  26.640350  46.355470  2.731807   

          rh       utci  
0  65.851090  20.411294  
1  67.107990  20.100860  
2  64.071020  21.698145  
3  62.181810  26.719557  
4  53.000023  30.037945  

Shape: (117936, 8)

Missing values:
time         0
latitude     0
longitude    0
tdb          0
mrt          0
v            0
rh           0
utci         0
dtype: int64


In [22]:
output_path = Path("data/processed/utci_analysis.csv")

# clean downstream analysis dataset
analysis_df.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print(f"Rows: {len(analysis_df):,}")
print(f"Columns: {analysis_df.columns.tolist()}")

Saved: data\processed\utci_analysis.csv
Rows: 117,936
Columns: ['time', 'latitude', 'longitude', 'tdb', 'mrt', 'v', 'rh', 'utci']


In [23]:
output_path = Path("data/processed/utci_hourly.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# full intermediate output file
utci_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(utci_df):,}")

Saved: data\processed\utci_hourly.csv
Rows: 117,936
